In [1]:
import pandas as pd
import numpy as np

In [2]:
# load and check the columns
import pandas as pd

meta = pd.read_csv("meta.csv", encoding="utf-8")
print(meta.columns)
meta.head(15)

Index(['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director',
       'genre', 'summary', 'awards', 'metascore', 'userscore', 'RelDate'],
      dtype='str')


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28
5,https://www.metacritic.com/movie/100-streets,100 Streets,Samuel Goldwyn Films,NaN,93.0,NaN,Jim O'Hanlon,Drama,NaN,NaN,44,6.1,2017-01-13
6,https://www.metacritic.com/movie/1000-times-go...,"1,000 Times Good Night",Film Movement,| Not Rated,117.0,NaN,Erik Poppe,Drama,NaN,NaN,57,6.8,2014-10-24
7,https://www.metacritic.com/movie/10000-bc,"10,000 BC",Warner Bros. Pictures,| PG-13,109.0,"Camilla Belle,Marco Khan,Steven Strait",Roland Emmerich,"Adventure,Drama,Fantasy",NaN,"#23MostDiscussedMovieof2008 , #27MostSharedMov...",34,4.6,2008-03-07
8,https://www.metacritic.com/movie/10000-km,"10,000 km",Broad Green Pictures,| R,99.0,NaN,Carlos Marques-Marcet,"Drama,Comedy,Romance","Two people in love, two apartments - one in Ba...",NaN,75,7.4,2015-07-10
9,https://www.metacritic.com/movie/1001-grams,1001 Grams,Kino Lorber,| Not Rated,93.0,NaN,Bent Hamer,Drama,When Norwegian scientist Marie attends a semin...,NaN,65,NaN,2015-05-08


In [3]:
# fix garbled names where needed
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

meta['director'] = meta['director'].dropna().apply(fix_encoding)

In [4]:
# split into one director per row and clean
director_long = meta['director'].dropna().str.split(',').explode().str.strip()

# drop empty strings (trailing commas) and leftover suffix fragments like "Jr."
director_long = director_long[~director_long.isin(['', 'Jr.', 'Sr.', 'II', 'III'])]
director_long.head(5)


0    Lynn Hershman-Leeson
1        Dan Trachtenberg
2         Brad Silberling
3            Jamie Linden
4         Cameron Cairnes
Name: director, dtype: str

In [5]:
# build table, sort A-Z, assign IDs
director = (
    director_long
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .to_frame(name='directorname')
    .reset_index(drop=True)
)
director.insert(0, 'directorid', director.index + 1)
director.head(5)

,directorid,directorname
0,1,A. Dean Bell
1,2,A.B. Shawky
2,3,A.J. Eaton
3,4,A.J. Edwards
4,5,Aaron Aites


In [6]:
# check for odd characters like before in Cast
print(director.shape)
print(director[director['directorname'].str.contains('Ã|Â', na=False)])  # should be empty
director.head(20)

(6136, 2)
Empty DataFrame
Columns: [directorid, directorname]
Index: []


,directorid,directorname
0,1,A. Dean Bell
1,2,A.B. Shawky
2,3,A.J. Eaton
3,4,A.J. Edwards
4,5,Aaron Aites
5,6,Aaron B. Koontz
6,7,Aaron Blaise
7,8,Aaron Brookner
8,9,Aaron Fisher
9,10,Aaron Harvey


In [7]:
# check for other weird characters
odd = sorted({c for name in director['directorname'] for c in name if ord(c) > 127})
print(odd)

['Á', 'Å', 'É', 'Ô', 'Ö', 'Ø', 'à', 'á', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ø', 'ú', 'û', 'ü']


In [8]:
# export
director.to_csv("director_extract.csv", index=False)